Goal:
Build a borrower-level probability-of-default model using
consumer lending data to support underwriting and portfolio
risk assessment decisions.

Prediction Timing:
The model uses only information available at loan origination.

Target:
Predict whether a loan ultimately defaults or is fully repaid.

## Load Data

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:\\OneDrive\\+Study\\C Projects\\fintech-credit-risk-model\\Data\\raw\\lendingclub.csv", low_memory=False)

df.shape
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.shape

(2260701, 151)

## Understand the loan status variable

In [15]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

## Define the target

In [ ]:
# bad loans
bad_status = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)"
]

In [ ]:
# good loans
good_status = [
    "Fully Paid"
]

In [18]:
df = df[df["loan_status"].isin(bad_status + good_status)]

df["target_default"] = np.where(
    df["loan_status"].isin(bad_status),
    1,
    0
)

C:\Users\59171\AppData\Local\Temp\ipykernel_7212\1826451357.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target_default"] = np.where(


## Initial leakage removal

In [19]:
leakage_cols = [
    "last_pymnt_d",
    "last_pymnt_amnt",
    "total_pymnt",
    "total_rec_prncp",
    "recoveries",
    "collection_recovery_fee",
    "out_prncp",
    "out_prncp_inv"
]

In [20]:
df = df.drop(columns=leakage_cols, errors="ignore")

## Select Initial Feature set

In [26]:
selected_features = [
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "issue_d",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "earliest_cr_line",
    "application_type",
    "addr_state",
    "target_default"
]

In [27]:
df = df[selected_features]

## Basic Missing value review

In [28]:
missing_pct = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
)

missing_pct.head(20)

emp_length             5.893962e-02
revol_util             6.468947e-04
dti                    2.939104e-04
inq_last_6mths         7.293063e-07
int_rate               0.000000e+00
term                   0.000000e+00
loan_amnt              0.000000e+00
home_ownership         0.000000e+00
sub_grade              0.000000e+00
grade                  0.000000e+00
verification_status    0.000000e+00
purpose                0.000000e+00
delinq_2yrs            0.000000e+00
issue_d                0.000000e+00
annual_inc             0.000000e+00
open_acc               0.000000e+00
pub_rec                0.000000e+00
revol_bal              0.000000e+00
total_acc              0.000000e+00
earliest_cr_line       0.000000e+00
dtype: float64

In [29]:
print(df["int_rate"].dtype)
print(df["int_rate"].head())

float64
0    13.99
1    11.99
2    10.78
4    22.45
5    13.44
Name: int_rate, dtype: float64


In [30]:
print(df["revol_util"].dtype)
print(df["revol_util"].head())

float64
0    29.7
1    19.2
2    56.2
4    64.5
5    68.4
Name: revol_util, dtype: float64


In [31]:
df["emp_length"].value_counts()

emp_length
10+ years    450068
2 years      124103
< 1 year     110521
3 years      109738
1 year        90224
5 years       85749
4 years       82225
6 years       63854
8 years       61621
7 years       60546
9 years       51701
Name: count, dtype: int64

## Save Processed Dataset

In [32]:
df.to_csv(
    "D:/OneDrive/+Study/C Projects/fintech-credit-risk-model/Data/processed/cleaned_loans.csv",
    index=False
)